# Father & Son: the parking dispute

Each agent sees the conversation from its own point of view: its own lines as its own, everyone else's labelled by name. A moderator then states what was agreed as structured data, and every turn is checked for an agent writing lines for someone else. Runs on any OpenAI-compatible model set in the repo's `.env`.

In [5]:
# Cell 1 — Setup
import json, os, sys
from datetime import date
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_BASE_URL") and os.getenv("OPENAI_API_KEY", "lm-studio") in ("", "lm-studio"):
    raise RuntimeError("No LLM endpoint: set OPENAI_BASE_URL (LM Studio) or a real OPENAI_API_KEY in the repo's .env")
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))
import conversation as cv

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
chat = cv.openai_chat()
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
print("Model:", MODEL)

Model: qwen/qwen3.6-35b-a3b


In [6]:
# Cell 2 — The scene and the agents
SCENE = "A father and his adult son talk in the kitchen about how the son parks in the shared driveway."
RUN_NAME = "father_son_run"
conv = cv.Conversation(SCENE, [
    cv.Agent("FATHER", "You're frustrated: the way your son parks leaves no room for the other cars. Be firm but respectful, and push for a concrete rule."),
    cv.Agent("SON", "You're polite and empathetic. Acknowledge the problem, ask clarifying questions, and offer specific solutions."),
])
OPENING = ("FATHER", "The way you park, I can't fit my car in the driveway. This keeps happening.")
ORDER, ROUNDS = ["SON", "FATHER"], 4

In [7]:
# Cell 3 — Run the conversation
log = cv.run(conv, chat, OPENING, ORDER, rounds=ROUNDS)
for t in log:
    flag = f"   [speaks for {', '.join(t['speaks_for_others'])}]" if t["speaks_for_others"] else ""
    print(f"{t['speaker']}: {t['text']}{flag}\n")

FATHER: The way you park, I can't fit my car in the driveway. This keeps happening.

SON: I completely understand how frustrating this has been, and I’m really sorry for the inconvenience. Could you let me know exactly where you’d prefer your car to be parked, or if there’s a specific time I should adjust my routine? I’d be happy to start backing in, using the street, or even painting a clear line on the driveway so we can both fit comfortably. Just let me know what works best for you, and I’ll make sure to stick to it.

FATHER: I appreciate you being willing to adjust, but we need a clear rule that works every time, not just when it’s convenient. From now on, you’ll always park on the right side of the driveway, leaving the left half completely clear for my car and our guests. Please mark that boundary with chalk today so we’re both on the same page.

SON: I completely understand and will make sure to park on the right side from now on to keep the left half clear for you and any guest

In [8]:
# Cell 4 — What did they agree, and did every agent stay in character?
summary = cv.summarize(conv, cv.openai_chat(temperature=0, max_tokens=400))
leaks = [t for t in log if t["speaks_for_others"]]
print(json.dumps(summary, indent=2))
print(f"\nTurns: {len(log)} | turns where an agent spoke for someone else: {len(leaks)}")

json.dump({"model": MODEL, "date": date.today().isoformat(), "scene": SCENE, "transcript": log,
           "summary": summary, "turns_speaking_for_others": len(leaks)},
          open(OUTPUT_DIR / f"{RUN_NAME}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(f"Saved outputs/{RUN_NAME}.json")

{
  "agreed": true,
  "terms": [
    "Son will park strictly on the right half of the driveway every day without exception",
    "The left half will remain clear for the father and guests",
    "Guests will park on the street",
    "A boundary line will be marked exactly down the center of the driveway",
    "The line will be painted using standard white or yellow permanent driveway marking paint",
    "The son will complete the painting tonight and send a photo upon completion"
  ],
  "open_issues": []
}

Turns: 9 | turns where an agent spoke for someone else: 0
Saved outputs/father_son_run.json
